# 📊 LIVR-Mini-Benchmark: GIAI ĐOẠN ĐÁNH GIÁ (EVALUATION MINI)
Notebook này thực hiện tinh chỉnh thích nghi và đánh giá hiệu năng mô hình **LIVR** đã được huấn luyện từ Notebook 1 trên **2 Novel Datasets** mới lạ hoàn toàn:
1. **VisuLogic**: Tác vụ đánh giá tư duy quy luật ma trận thị giác trừu tượng.
2. **CV-Bench**: Tác vụ đánh giá kỹ năng ước lượng độ sâu không gian và đếm.

### Nội dung chính:
1. **Đồng bộ mã nguồn**: Pull code mới nhất từ nhánh `develop` từ Github.
2. **Đọc cấu hình & Tải dataset**: Đọc file `evaluation_config.json` và chuẩn bị các phân đoạn tập thử nghiệm từ Hugging Face.
3. **Tái dựng mô hình**: Load base model Qwen2.5-VL-3B-Instruct dạng 4-bit, khôi phục LoRA adapter và nhúng của các Latent Tokens đã học.
4. **Tinh chỉnh thích nghi ngắn hạn**: Huấn luyện thích ứng (domain-specific fine-tuning) trên tập train của Novel Datasets với 2 Epochs ở Stage 2 (Image Visible).
5. **Đánh giá kiểm định khoa học (Sanity Check)**:
   - Đo lường chỉ số **Top-1 Accuracy** ở Stage 2 (Mở mắt - có ảnh).
   - Chặn thông tin ảnh đột ngột ở Stage 1 (Bịt mắt - không nhìn ảnh trực tiếp, chỉ dùng Latent Tokens) để kiểm định khả năng lưu trữ thông tin thị giác của các Latent Tokens.

In [ ]:
# =========================================================================
# CELL 1: KẾT NỐI GOOGLE DRIVE & ĐỒNG BỘ CODE TỪ GITHUB (DEVELOP BRANCH)
# =========================================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Cấu hình URL repository của bạn
REPO_URL = "https://github.com/CodeDaoVietNam/LIVR-Mini-Benchmark.git"
PROJECT_DIR = "LIVR-Mini-Benchmark"
BRANCH = "develop"

%cd /content
import os
if not os.path.exists(PROJECT_DIR):
    print(f"---> Đang thực hiện clone repo {REPO_URL} (nhánh {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
    %cd {PROJECT_DIR}
else:
    print(f"---> Repo {PROJECT_DIR} đã tồn tại. Đang tiến hành pull code mới nhất từ nhánh {BRANCH}...")
    %cd {PROJECT_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

In [ ]:
# =========================================================================
# CELL 2: CÀI ĐẶT THƯ VIỆN & PHÁT HIỆN GPU
# =========================================================================
# Cài đặt các thư viện lõi từ requirements.txt
!pip install -r requirements.txt

import sys
import os
# Đảm bảo Python nhận diện được các module trong thư mục src/
sys.path.append(os.getcwd())

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# =========================================================================
# CELL 3: NẠP FILE CẤU HÌNH ĐÁNH GIÁ
# =========================================================================
import json

with open("config/evaluation_config.json", "r", encoding="utf-8") as f:
    eval_config = json.load(f)

# Đổi từ visu_logic sang math_vista và override train/test samples cho Colab / Local
if 'visu_logic' in eval_config['eval_datasets']:
    eval_config['eval_datasets']['math_vista'] = eval_config['eval_datasets'].pop('visu_logic')
    eval_config['eval_datasets']['math_vista']['name'] = "MathVista"
    eval_config['eval_datasets']['math_vista']['huggingface_path'] = "AI4Math/MathVista"
    
eval_config['eval_datasets']['math_vista']['train_samples'] = 800
eval_config['eval_datasets']['math_vista']['test_samples'] = 200
eval_config['eval_datasets']['cv_bench']['train_samples'] = 800
eval_config['eval_datasets']['cv_bench']['test_samples'] = 200

print("COLAB/LOCAL EVALUATION CONFIGURATION:")
print(json.dumps(eval_config, indent=2))

In [ ]:
# =========================================================================
# CELL 4: TẢI NOVEL DATASETS (MATHVISTA & CV-BENCH)
# =========================================================================
from datasets import load_dataset
import os

cache_dir = None
if os.path.exists("/content"):
    cache_dir = "/content/dataset_cache"
    os.makedirs(cache_dir, exist_ok=True)
    print(f"-> Sử dụng ổ SSD cục bộ của Colab để lưu cache dataset: {cache_dir}")
else:
    cache_dir = os.path.join(os.getcwd(), "dataset_cache")
    os.makedirs(cache_dir, exist_ok=True)

print("---> Đang tải tập dữ liệu MathVista (split testmini có ảnh)... ")
try:
    mathvista_dataset = load_dataset("AI4Math/MathVista", split="testmini", cache_dir=cache_dir)
    print("MathVista testmini Dataset:", mathvista_dataset)
except Exception as e:
    print(f"Lỗi tải MathVista: {e}.")
    mathvista_dataset = None

print("\n---> Đang tải tập dữ liệu CV-Bench...")
try:
    cv_dataset = load_dataset("nyu-visionx/CV-Bench", cache_dir=cache_dir)
    print("CV-Bench Dataset:", cv_dataset)
except Exception as e:
    print(f"Lỗi tải CV-Bench: {e}.")
    cv_dataset = None

In [ ]:
# =========================================================================
# CELL 5: LOAD BASE MODEL & KHÔI PHỤC CHECKPOINT HUẤN LUYỆN TỪ NOTEBOOK 1
# =========================================================================
import os
import torch
from src.model import LIVRModelManager
from src.mask import patch_model_for_livr

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = eval_config["checkpoint_path"]

# 1. Load base model dạng 4-bit giúp tối ưu VRAM cho card T4
manager = LIVRModelManager(
    model_id=eval_config["base_model_id"],
    K=eval_config["K"],
    device=device,
    load_in_4bit=True
)

# 2. Khởi tạo LoRA adapters
model = manager.setup_peft_and_freezing()

# 3. Nạp trọng số checkpoint đã học ở Notebook 1
LOAD_IMPLEMENT_CHECKPOINT = os.path.exists(checkpoint_path)
if LOAD_IMPLEMENT_CHECKPOINT:
    print(f"---> Đang khôi phục trọng số huấn luyện từ: {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Khôi phục thủ công vector biểu diễn của Latent Tokens vào Embedding Layer
    with torch.no_grad():
        model.base_model.model.model.embed_tokens.weight[manager.latent_token_ids] = checkpoint['latent_embeddings'].to(device)
    print("---> Đã khôi phục thành công toàn bộ mô hình và vector Latent Tokens!")
else:
    print(f"[CẢNH BÁO] Không tìm thấy file checkpoint tại {checkpoint_path}.")

# 4. Monkey-patch Custom Attention Mask
patch_model_for_livr(
    model=model,
    latent_token_ids=manager.latent_token_ids,
    image_pad_token_id=manager.image_pad_token_id,
    pad_token_id=manager.pad_token_id
)
processor = manager.processor

In [ ]:
# =========================================================================
# CELL 6: HUÂN LUYỆN TINH CHỈNH THÍCH NGHI ĐA MIỀN TRI THỨC (STAGE 2)
# =========================================================================
import os
import torch
import gc
import copy
from PIL import Image
from torch.optim import AdamW
from tqdm import tqdm
from torch.cuda.amp import GradScaler

# Định nghĩa cục bộ prepare_vqa_inputs trực tiếp trong notebook
def prepare_vqa_inputs(processor, conversation, latent_tokens, device="cuda"):
    conv = copy.deepcopy(conversation)
    latent_str = "".join(latent_tokens)
    for msg in conv:
        if msg["role"] == "user":
            for content_item in msg["content"]:
                if content_item["type"] == "text":
                    content_item["text"] = f"{content_item['text'].strip()}\n{latent_str}"
    is_training = (conv[-1]["role"] == "assistant")
    full_text = processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=not is_training)
    user_conv = [msg for msg in conv if msg["role"] == "user"]
    prompt_text = processor.apply_chat_template(user_conv, tokenize=False, add_generation_prompt=True)
    images = []
    for msg in conv:
        if msg["role"] == "user":
            for content_item in msg["content"]:
                if content_item["type"] == "image":
                    img_data = content_item["image"]
                    if img_data is not None:
                        if isinstance(img_data, str):
                            img_data = Image.open(img_data).convert("RGB")
                            content_item["image"] = img_data
                        images.append(img_data)
    images_arg = [images] if len(images) > 0 else None
    full_inputs = processor(text=[full_text], images=images_arg, padding=True, return_tensors="pt")
    prompt_inputs = processor(text=[prompt_text], images=images_arg, padding=True, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in full_inputs.items()}
    labels = inputs["input_ids"].clone()
    prompt_len = prompt_inputs["input_ids"].size(1)
    labels[:, :prompt_len] = -100
    inputs["labels"] = labels
    return inputs

model.train()
model.livr_stage = 2

lr = eval_config.get("learning_rate", 5e-5)
if LOAD_IMPLEMENT_CHECKPOINT:
    stage1_epochs = 0
    epochs = eval_config.get("fine_tune_epochs", 2)
else:
    stage1_epochs = 2
    epochs = stage1_epochs + eval_config.get("fine_tune_epochs", 3)
grad_accum_steps = eval_config.get("grad_accumulation_steps", 8)
    output_dir = eval_config.get("output_dir", "/content/drive/MyDrive/LIVR_Mini_Project/checkpoints/evaluation")
os.makedirs(output_dir, exist_ok=True)
    cache_dir = "/content/dataset_cache" if os.path.exists("/content") else os.path.join(os.getcwd(), "dataset_cache")

print("---> Đang chuẩn bị dữ liệu tinh chỉnh thích nghi...")

# Hàm chuẩn bị dữ liệu hội thoại từ HuggingFace datasets có xử lý split và chia tách động
def prepare_eval_dataset(hf_dataset, num_samples, is_train=True):
    if hf_dataset is None:
        return []
    data_list = []
    
    from datasets import DatasetDict
    if isinstance(hf_dataset, DatasetDict):
        if 'train' in hf_dataset:
            split_name = 'train'
        elif 'test' in hf_dataset:
            split_name = 'test'
        else:
            split_name = list(hf_dataset.keys())[0]
        dataset_split = hf_dataset[split_name]
        is_single_split = (split_name == 'test')
    else:
        # Đối với dataset đơn lẻ như MathVista testmini
        dataset_split = hf_dataset
        is_single_split = True
    
    # Chia tách động để tránh rò rỉ dữ liệu (data leakage) trên các tập đơn split
    if is_single_split and is_train:
        split_data = dataset_split.select(range(min(num_samples, len(dataset_split))))
    elif is_single_split and not is_train:
        start_idx = max(0, len(dataset_split) - num_samples)
        split_data = dataset_split.select(range(start_idx, len(dataset_split)))
    else:
        split_data = dataset_split.select(range(min(num_samples, len(dataset_split))))
        
    for item in split_data:
        # Trích xuất ảnh (MathVista dùng decoded_image hoặc image)
        image_obj = item.get('decoded_image') or item.get('image')
        if isinstance(image_obj, str) and image_obj:
            img_path = os.path.join(cache_dir, image_obj)
            if os.path.exists(img_path):
                from PIL import Image
                image_obj = Image.open(img_path).convert("RGB")
        elif image_obj is not None:
            from PIL import Image
            if isinstance(image_obj, Image.Image):
                image_obj = image_obj.convert("RGB")
        
        prompt = item.get('prompt', item.get('question', item.get('query', '')))
        answer = str(item.get('answer', item.get('label', ''))).strip()
        choices = item.get('choices', None)
        
        if choices and isinstance(choices, list):
            if "(A)" not in prompt:
                options_str = " ".join([f"({chr(65+idx)}) {opt}" for idx, opt in enumerate(choices)])
                prompt = f"{prompt}\nSelect from the following choices:\n{options_str}\nAnswer with the option's letter directly."
            else:
                if "letter" not in prompt.lower():
                    prompt = f"{prompt}\nAnswer with the option's letter directly."
        
        # Chỉ chèn thẻ image khi có ảnh thực sự
        user_content = []
        if image_obj is not None:
            user_content.append({"type": "image", "image": image_obj})
        user_content.append({"type": "text", "text": prompt})
        
        formatted_conv = [
            {
                "role": "user",
                "content": user_content
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": answer}
                ]
            }
        ]
        data_list.append({"conversation": formatted_conv})
    return data_list

# Chuẩn bị dữ liệu thích ứng (dùng mathvista_dataset và cv_dataset)
mathvista_train = prepare_eval_dataset(mathvista_dataset, eval_config['eval_datasets']['math_vista']['train_samples'], is_train=True)
cv_train = prepare_eval_dataset(cv_dataset, eval_config['eval_datasets']['cv_bench']['train_samples'], is_train=True)
combined_train = mathvista_train + cv_train

print(f"Tổng số mẫu tinh chỉnh thích nghi: {len(combined_train)} mẫu")

if len(combined_train) > 0:
    gc.collect()
    torch.cuda.empty_cache()
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    scaler = GradScaler()
    
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        optimizer.zero_grad()
        # Xác định stage hiện tại
        if epoch <= stage1_epochs:
            current_stage = 1
            model.livr_stage = 1
        else:
            current_stage = 2
            model.livr_stage = 2
            
        progress_bar = tqdm(combined_train, desc=f"Adaptation [Stage {current_stage}] Epoch {epoch}/{epochs}")
        
        for step, batch in enumerate(progress_bar):
            try:
                inputs = prepare_vqa_inputs(
                    processor=processor,
                    conversation=batch['conversation'],
                    latent_tokens=manager.latent_tokens,
                    device="cuda"
                )
                
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    outputs = model(**inputs)
                    loss = outputs.loss / grad_accum_steps
                
                scaler.scale(loss).backward()
                epoch_loss += loss.item() * grad_accum_steps
                
                if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(combined_train):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=0.5)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    
                progress_bar.set_postfix({"Loss": f"{loss.item() * grad_accum_steps:.4f}"})
                del inputs, outputs, loss
                if step % 10 == 0:
                    gc.collect()
                    torch.cuda.empty_cache()
                    
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print("\n[WARNING] Bắt gặp lỗi OOM, đang dọn cache CUDA và bỏ qua bước này...")
                    optimizer.zero_grad()
                    del e
                    gc.collect()
                    torch.cuda.empty_cache()
                    continue
                else:
                    raise e
            
        print(f"➔ Kết thúc Epoch {epoch} - Average Loss: {epoch_loss / len(combined_train):.4f}")
        
    ft_checkpoint_path = os.path.join(output_dir, "livr_eval_finetuned.pt")
    trainable_names = {n for n, p in model.named_parameters() if p.requires_grad}
    trainable_sd = {k: v.cpu() for k, v in model.state_dict().items() if k in trainable_names}
    
    torch.save({
        'model_state_dict': trainable_sd,
        'latent_embeddings': model.get_input_embeddings().weight[manager.latent_token_ids].detach().cpu()
    }, ft_checkpoint_path)
    print(f"---> Đã lưu checkpoint thích ứng thành công tại: {ft_checkpoint_path}")
else:
    print("[CẢNH BÁO] Không tìm thấy dữ liệu thích ứng để tinh chỉnh.")


In [ ]:
# =========================================================================
# CELL 7: ĐÁNH GIÁ ĐỘ CHÍNH XÁC ACCURACY & KIỂM ĐỊNH KHOA HỌC (SANITY CHECK)
# =========================================================================
import re
import copy
from PIL import Image

# Định nghĩa cục bộ prepare_vqa_inputs trực tiếp trong notebook Cell 7
def prepare_vqa_inputs(processor, conversation, latent_tokens, device="cuda"):
    conv = copy.deepcopy(conversation)
    latent_str = "".join(latent_tokens)
    for msg in conv:
        if msg["role"] == "user":
            for content_item in msg["content"]:
                if content_item["type"] == "text":
                    content_item["text"] = f"{content_item['text'].strip()}\n{latent_str}"
    is_training = (conv[-1]["role"] == "assistant")
    full_text = processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=not is_training)
    user_conv = [msg for msg in conv if msg["role"] == "user"]
    prompt_text = processor.apply_chat_template(user_conv, tokenize=False, add_generation_prompt=True)
    images = []
    for msg in conv:
        if msg["role"] == "user":
            for content_item in msg["content"]:
                if content_item["type"] == "image":
                    img_data = content_item["image"]
                    if img_data is not None:
                        if isinstance(img_data, str):
                            img_data = Image.open(img_data).convert("RGB")
                            content_item["image"] = img_data
                        images.append(img_data)
    images_arg = [images] if len(images) > 0 else None
    full_inputs = processor(text=[full_text], images=images_arg, padding=True, return_tensors="pt")
    prompt_inputs = processor(text=[prompt_text], images=images_arg, padding=True, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in full_inputs.items()}
    labels = inputs["input_ids"].clone()
    prompt_len = prompt_inputs["input_ids"].size(1)
    labels[:, :prompt_len] = -100
    inputs["labels"] = labels
    return inputs

# Hàm so khớp câu trả lời thông minh tránh lệch định dạng (ví dụ: 'A', '(A)', 'option A')
def match_answer(pred, target):
    pred = pred.strip().lower()
    target = target.strip().lower()
    if pred == target:
        return True
    pred_cleaned = pred.replace('(', '').replace(')', '').replace('.', '').strip()
    target_cleaned = target.replace('(', '').replace(')', '').replace('.', '').strip()
    if pred_cleaned == target_cleaned:
        return True
    if len(target_cleaned) == 1 and target_cleaned.isalpha():
        pattern = rf"\b{target_cleaned}\b"
        if re.search(pattern, pred_cleaned):
            return True
    return False

def evaluate_accuracy(model, eval_data, manager, name="MathVista", max_samples=100):
    model.eval()
    correct = 0
    total = 0
    log_entries = []
    cache_dir = "/content/dataset_cache" if os.path.exists("/content") else os.path.join(os.getcwd(), "dataset_cache")
    
    print(f"\n➔ Đang chạy đánh giá trên {name} (Giới hạn {max_samples} mẫu)...")
    with torch.no_grad():
        for i, item in enumerate(eval_data):
            if i >= max_samples:
                break
                
            # Trích xuất ảnh (MathVista dùng decoded_image hoặc image)
            image = item.get('decoded_image') or item.get('image')
            if isinstance(image, str) and image:
                img_path = os.path.join(cache_dir, image)
                if os.path.exists(img_path):
                    from PIL import Image
                    image = Image.open(img_path).convert("RGB")
            elif image is not None:
                from PIL import Image
                if isinstance(image, Image.Image):
                    image = image.convert("RGB")
            
            prompt = item.get('prompt', item.get('question', item.get('query', 'How many objects are there in this image?')))
            target = str(item.get('answer', item.get('label', ''))).strip()
            choices = item.get('choices', None)
            
            if choices and isinstance(choices, list):
                if "(A)" not in prompt:
                    options_str = " ".join([f"({chr(65+idx)}) {opt}" for idx, opt in enumerate(choices)])
                    prompt = f"{prompt}\nSelect from the following choices:\n{options_str}\nAnswer with the option's letter directly."
                else:
                    if "letter" not in prompt.lower():
                        prompt = f"{prompt}\nAnswer with the option's letter directly."
            
            # Chỉ chèn thẻ image khi có ảnh thực sự
            user_content = []
            if image is not None:
                user_content.append({"type": "image", "image": image})
            user_content.append({"type": "text", "text": prompt})
            
            conv = [
                {
                    "role": "user",
                    "content": user_content
                }
            ]
            
            inputs = prepare_vqa_inputs(
                processor=manager.processor,
                conversation=conv,
                latent_tokens=manager.latent_tokens,
                device="cuda"
            )
            inputs.pop("labels", None)
            
            outputs = model.generate(**inputs, max_new_tokens=10)
            input_len = inputs["input_ids"].shape[1]
            pred_text = manager.processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
            
            is_correct = match_answer(pred_text, target)
            if is_correct:
                correct += 1
            total += 1
            
            log_entries.append({
                "index": i + 1,
                "question": prompt,
                "ground_truth": target,
                "model_prediction": pred_text,
                "is_correct": is_correct
            })
            
            if i < 5:
                print(f"   [Mẫu {i+1}] Hỏi: {prompt[:80]}... | Đúng: {target} | Đoán: {pred_text} | Kết quả: {'ĐÚNG' if is_correct else 'SAI'}")
            
    accuracy = (correct / total) * 100 if total > 0 else 0.0
    print(f"[{name}] Accuracy: {accuracy:.2f}% ({correct}/{total})")
    
    name_slug = re.sub(r'[^a-zA-Z0-9_]', '_', name.lower().strip())
    log_path = os.path.join(eval_config["output_dir"], f"eval_details_{name_slug}.json")
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, "w", encoding="utf-8") as lf:
        json.dump(log_entries, lf, ensure_ascii=False, indent=2)
    print(f"   ➔ Đã lưu nhật ký chi tiết của {name} tại: {log_path}")
    
    return accuracy

print("=== BẮT ĐẦU ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH ===")
results = {}

if cv_dataset is not None:
    test_data = cv_dataset['test']
    num_test = eval_config['eval_datasets']['cv_bench'].get('test_samples', 100)
    test_data_slice = test_data.select(range(max(0, len(test_data) - num_test), len(test_data)))
    
    model.livr_stage = 2
    acc_stage2 = evaluate_accuracy(model, test_data_slice, manager, name="CV-Bench Stage 2 (Mở mắt)", max_samples=num_test)
    
    model.livr_stage = 1
    acc_stage1 = evaluate_accuracy(model, test_data_slice, manager, name="CV-Bench Stage 1 (Bịt mắt - Sanity Check)", max_samples=num_test)
    
    results["CV-Bench"] = {
        "Stage 2 (Mở)": acc_stage2,
        "Stage 1 (Bịt - Sanity Check)": acc_stage1,
        "Sụt giảm": acc_stage2 - acc_stage1
    }

if mathvista_dataset is not None:
    num_test = eval_config['eval_datasets']['math_vista'].get('test_samples', 100)
    test_data_slice = mathvista_dataset.select(range(max(0, len(mathvista_dataset) - num_test), len(mathvista_dataset)))
    
    model.livr_stage = 2
    acc_stage2 = evaluate_accuracy(model, test_data_slice, manager, name="MathVista Stage 2 (Mở mắt)", max_samples=num_test)
    
    model.livr_stage = 1
    acc_stage1 = evaluate_accuracy(model, test_data_slice, manager, name="MathVista Stage 1 (Bịt mắt - Sanity Check)", max_samples=num_test)
    
    results["MathVista"] = {
        "Stage 2 (Mở)": acc_stage2,
        "Stage 1 (Bịt - Sanity Check)": acc_stage1,
        "Sụt giảm": acc_stage2 - acc_stage1
    }

print("\n" + "="*70)
print(" BẢNG TỔNG KẾT HIỆU NĂNG & KIỂM ĐỊNH KHOA HỌC (SANITY CHECK)")
print("="*70)
print(f"{'Dataset':<15} | {'Stage 2 (Mở)':<15} | {'Stage 1 (Bịt)':<20} | {'Sụt giảm':<10}")
print("-"*70)
for ds_name, metrics in results.items():
    print(f"{ds_name:<15} | {metrics['Stage 2 (Mở)']:>13.2f}% | {metrics['Stage 1 (Bịt - Sanity Check)']:>18.2f}% | {metrics['Sụt giảm']:>8.2f}%")
print("="*70)
print("Giải nghĩa khoa học:")
print("1. Stage 2 (Mở mắt): Đo lường khả năng giải quyết tác vụ khi ảnh hiển thị đầy đủ.")
print("2. Stage 1 (Bịt mắt): Chặn ảnh hoàn toàn. Mô hình bắt buộc phải trả lời dựa trên thông tin tích lũy")
print("   trong Latent Tokens.")
print("3. Mức sụt giảm vừa phải chứng minh Latent Tokens đóng vai trò là một 'hộp đen' thị giác xuất sắc!")